# Notebook Setup & Imports

In [15]:
import sys
from pathlib import Path

# Add project root to PYTHONPATH
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import torch
import pandas as pd
import numpy as np

# Import Project Modules

In [16]:
from src.dataset import load_data, make_data_loader, make_windows
from src.model import StockMLP, StockCNN
from src.train import train
from src.evaluate import evaluate

# Configuration

In [17]:
DATA_PATH = PROJECT_ROOT / "data" / "train.csv"
BATCH_SIZE = 128
WINDOW_SIZE = 30
EPOCHS = 100
USE_CNN = False  # switch between MLP and CNN
NORMALIZE = True

# Load Dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Please place train.csv inside data/ directory."
    )

df = load_data(DATA_PATH, frac=0.01, random_state=42)
print(f"Rows loaded: {len(df):,}")
# load_data already samples the rows
df.describe()


# Creating Sliding Windows

In [ ]:
X, y = make_windows(df, window=WINDOW_SIZE)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive ratio:", y.mean())


X shape: (209960, 30)
y shape: (209960,)
Positive ratio: 0.4999190321966089


# Train / Validation Split

In [ ]:
split_idx = int(len(X) * 0.8)

X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"Total samples: {len(X):,}")
print(f"Train samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")


Total samples: 209,960
Train samples: 167,968
Validation samples: 41,992


# DataLoaders

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

if NORMALIZE:
    from src.features import normalize
    X_train = normalize(X_train)
    X_val = normalize(X_val)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Initialize Model

In [ ]:
if USE_CNN:
    model = StockCNN()
    print("Using CNN model")
else:
    model = StockMLP(input_dim=WINDOW_SIZE)
    print("Using MLP model")

model

Using MLP model


StockMLP(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

# Train Model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()
print("hello")
train(
    model=model,
    loader=train_loader,
    epochs=EPOCHS,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
)


hello
Epoch 1/10 - Loss: 0.5520
Epoch 2/10 - Loss: 0.5304
Epoch 3/10 - Loss: 0.5272
Epoch 4/10 - Loss: 0.5261
Epoch 5/10 - Loss: 0.5240
Epoch 6/10 - Loss: 0.5224
Epoch 7/10 - Loss: 0.5208
Epoch 8/10 - Loss: 0.5195
Epoch 9/10 - Loss: 0.5187
Epoch 10/10 - Loss: 0.5180


# Evaluate on Validation Set

In [ ]:
model.to(device)

val_accuracy = evaluate(
    model,
    X_val,
    y_val,
    device=device
)

print(f"Validation Accuracy: {val_accuracy:.4f}")


Validation Accuracy: 0.7448


# Save Trained Model

In [ ]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "stock_model_v1.pt"
torch.save(model.state_dict(), model_path)

print(f"Model saved to: {model_path}")

Model saved to: /home/mega/projects/forth-year/NN/stock-trend/models/stock_model_v1.pt


# Quick Sanity Prediction

In [ ]:
model.eval()

sample = torch.tensor(X_val[:5], dtype=torch.float32).to(device)
with torch.no_grad():
    logits = model(sample)
    probs = torch.sigmoid(logits)

print("Predicted probabilities:", probs.cpu().numpy())
print("True labels:", y_val[:5])

Predicted probabilities: [0.15563229 0.5636733  0.5708137  0.12137985 0.9367694 ]
True labels: [0 0 1 0 1]
